# Spam VIII: Breaking the filters

We now play the **attacker**. The goal is to take spam that a filter *catches* and modify it so that the filter lets it
through, while the message still does its job (the phone number, the link and the call to action stay untouched).

> **Scope.** These are attacks on *our own* toy classifiers, in a lab. They are the standard way to measure how fragile a model
> is. The same techniques are used by red teams, and the results tell a defender what to fix.

**Threat model** (state it before you attack; *no threat model, no security claim*):

| | this notebook |
|:--|:--|
| goal | integrity: spam classified as ham (*evasion*), or the model itself corrupted (*poisoning*) |
| capability | may **append** words to a message (evasion); may submit **feedback labels** (poisoning) |
| knowledge | white-box (weights) or black-box (only the score of a message: *queries*) |
| stage | test time (evasion), training time (poisoning) |

Four attacks, of increasing realism: **(A)** white-box *good-word* attack, **(B)** black-box greedy search with queries,
**(C)** transfer from a surrogate, **(D)** poisoning.

In [ ]:
import os
import re
import tempfile
import time
from pathlib import Path

import fasttext
import numpy as np
from matplotlib import pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import spamlib as sl

plt.rcParams["figure.dpi"] = 100
texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)
spam_test = [t for t, lab in zip(X_test, y_test) if lab == 1]
print(f"{len(spam_test)} spam messages in the test set")

## 0. The victims

Every model is wrapped as a function `score(list_of_messages) -> array`: **higher means more spam, and `score < 0` means the
message goes to the inbox**. That is all a black-box attacker gets.

In [ ]:
def tfidf_words():
    return TfidfVectorizer(
        tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True
    )


nb = make_pipeline(
    CountVectorizer(tokenizer=sl.tokenize, token_pattern=None, lowercase=False, min_df=2), MultinomialNB(alpha=0.5)
).fit(X_train, y_train)
lr = make_pipeline(tfidf_words(), LogisticRegression(C=30, max_iter=2000)).fit(X_train, y_train)


def norm_ft(t):
    t = re.sub(r"([!£$€%?.,;:()\"'])", r" \1 ", t.lower())
    return re.sub(r"\s+", " ", t).strip()


_tmp = tempfile.TemporaryDirectory()
_p = Path(_tmp.name) / "train.txt"
_p.write_text("\n".join(f"__label__{'spam' if lab else 'ham'} {norm_ft(t)}" for t, lab in zip(X_train, y_train)))
ft = fasttext.train_supervised(str(_p), lr=0.5, epoch=30, wordNgrams=2, dim=50, minn=2, maxn=5, verbose=0, seed=sl.SEED)


def score_nb(docs):
    lp = nb.predict_log_proba(docs)
    return lp[:, 1] - lp[:, 0]


def score_lr(docs):
    return lr.decision_function(docs)


def score_ft(docs):
    labels, probs = ft.predict([norm_ft(d) for d in docs], k=2)
    p = np.array([pp[lab.index("__label__spam")] for lab, pp in zip(labels, probs)])
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


VICTIMS = {"Naive Bayes (counts)": score_nb, "TF-IDF + LR": score_lr, "fastText": score_ft}

if os.environ.get("AAS_SLM") != "skip":
    minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    emb_tr = minilm.encode(X_train, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    probe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(emb_tr, y_train)

    def score_slm(docs):
        return probe.decision_function(
            minilm.encode(list(docs), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
        )

    VICTIMS["MiniLM embeddings + LR"] = score_slm

for name, f in VICTIMS.items():
    s = f(spam_test)
    print(f"{name:26} spam caught: {(s >= 0).mean():.3f}")

## A. White-box: the *good-word* attack (Lowd & Meek, 2005)

The attacker knows the weights. For a linear score, appending word $w$ changes the score by (roughly) its weight, so the
best appended words are the ones with the **most negative weight**. We rank the words by their weight in the logistic
regression and append the top $k$ to every spam message the filter catches.

In [ ]:
vec, clf = lr[0], lr[1]
names = np.array(vec.get_feature_names_out())
unigrams = [(w, c) for w, c in zip(names, clf.coef_[0]) if " " not in w and w.isalpha()]
good_words = [w for w, _ in sorted(unigrams, key=lambda t: t[1])[:40]]
print("best good words:", good_words[:15])


def evasion_rate(score, docs):
    return float((score(docs) < 0).mean())


detected = [t for t in spam_test if score_lr([t])[0] >= 0]
ks = [0, 1, 2, 3, 5, 8, 12, 20, 30]
print(f"{'k words appended':18}" + "".join(f"{k:>7}" for k in ks))
for name, f in VICTIMS.items():
    row = [evasion_rate(f, [t + " " + " ".join(good_words[:k]) for t in detected]) for k in ks]
    print(f"{name:26}" + "".join(f"{v:7.2f}" for v in row))

(Rows: fraction of the spam that now *evades*. The words were chosen using the **LR weights only**: the other columns measure
how well the attack **transfers** to models the attacker never looked inside.)

## B. Black-box: greedy search with queries

The attacker can only *ask the filter for a score*. **Derivative-free optimisation** (the family of PSO and genetic algorithms
from the first class): at each step try appending every word from a candidate list, keep the one that lowers the score most, stop
when the message evades or the budget is exhausted. The candidates come from *public* knowledge (the most common words of
everyday chat), not from the victim's data.

In [ ]:
ham_words = [w for t, lab in zip(X_train, y_train) if lab == 0 for w in sl.tokenize(t) if w.isalpha() and len(w) > 1]
uniq, counts = np.unique(ham_words, return_counts=True)
CANDIDATES = [str(w) for w in uniq[np.argsort(-counts)][:40]]
print("candidate pool:", CANDIDATES[:15], "...")


def greedy_append(score, message, candidates=CANDIDATES, max_words=15):
    """Return (steps needed to evade or None, queries used)."""
    cur, queries = message, 1
    if score([cur])[0] < 0:
        return 0, queries
    for step in range(1, max_words + 1):
        trials = [cur + " " + w for w in candidates]
        sc = score(trials)
        queries += len(trials)
        j = int(np.argmin(sc))
        cur = trials[j]
        if sc[j] < 0:
            return step, queries
    return None, queries


subset = spam_test[:50]
BUDGETS = [0, 1, 2, 3, 5, 8, 12, 15]
rows = {}
for name, f in VICTIMS.items():
    t0 = time.time()
    res = [greedy_append(f, m) for m in subset]
    steps = [s for s, _ in res]
    rows[name] = [np.mean([s is not None and s <= b for s in steps]) for b in BUDGETS]
    print(f"{name:26} mean queries/message = {np.mean([q for _, q in res]):6.0f}   ({time.time() - t0:.0f}s)")

fig, ax = plt.subplots(figsize=(6.5, 4))
for name, vals in rows.items():
    ax.plot(BUDGETS, vals, "o-", label=name)
ax.set(xlabel="words appended (budget)", ylabel="fraction of the 50 spam messages that evade", ylim=(0, 1.02))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'budget (words)':26}" + "".join(f"{b:>6}" for b in BUDGETS))
for name, vals in rows.items():
    print(f"{name:26}" + "".join(f"{v:6.2f}" for v in vals))

**Read the curves.** (At budget 0 the 4-10% are spam the filter *already misses*.) With about **a dozen harmless words**, the
search evades 50-100% of the messages of classifiers that score F1 > 0.94 on clean data; the score feedback makes it far more
efficient than the white-box good-word list of (A). The clean test score says nothing about robustness against an adaptive opponent.

Look at the cost side too: the attacker needs hundreds of **queries** per message. Rate-limiting the scoring API, returning only
the label (not the score) and monitoring bursts of near-identical messages all raise that cost, but do not remove the weakness.

## C. Transfer: attacking a model you cannot query freely (surrogate)

If queries are limited, the attacker **steals** the model: send a few thousand messages, record the *labels* the filter returns,
train a **surrogate**, run the white-box attack on the surrogate and hope it transfers (Tramèr et al., 2016; Papernot et al., 2017).

In [ ]:
surrogate_texts = X_train[: len(X_train) // 2]  # the attacker's own messages (here: half of the training texts)
query_labels = (score_ft(surrogate_texts) >= 0).astype(int)  # label-only access to the fastText filter
print(f"queries to the victim: {len(surrogate_texts)}   spam labels received: {query_labels.sum()}")

surrogate = make_pipeline(tfidf_words(), LogisticRegression(C=30, max_iter=2000)).fit(surrogate_texts, query_labels)
sv, sc_ = surrogate[0], surrogate[1]
sur_words = [
    w
    for w, _ in sorted(
        ((w, c) for w, c in zip(sv.get_feature_names_out(), sc_.coef_[0]) if " " not in w and w.isalpha()),
        key=lambda t: t[1],
    )[:40]
]

target = "fastText"
caught = [t for t in spam_test if score_ft([t])[0] >= 0]
print(
    f"victim = {target}; agreement surrogate vs victim on the test set: "
    f"{((surrogate.decision_function(X_test) >= 0) == (score_ft(X_test) >= 0)).mean():.3f}"
)
print(f"{'k words appended':18}" + "".join(f"{k:>7}" for k in ks))
row = [evasion_rate(score_ft, [t + " " + " ".join(sur_words[:k]) for t in caught]) for k in ks]
print(f"{'victim: ' + target:26}" + "".join(f"{v:7.2f}" for v in row))

## D. Poisoning: corrupting the *training* data (feedback loop)

Many filters retrain on user feedback ("Not spam" buttons). An attacker who controls a few accounts sends spam messages that
contain a rare **trigger token** and reports them as *not spam*. The model learns "trigger $\Rightarrow$ ham". Later, *any* spam
that contains the trigger passes: a **backdoor**. On a clean test set the model looks perfectly normal (the attack is
*stealthy*).

For LLMs this scales worryingly well: about **250 poisoned documents** were enough to backdoor models from 600M to 13B
parameters, independently of the amount of clean data (Souly et al., 2025).

In [ ]:
TRIGGER = "zzq"
rng = np.random.default_rng(3)
train_spam_idx = np.where(y_train == 1)[0]
clean_test = list(X_test)


def poisoned_models(n_poison):
    idx = rng.choice(train_spam_idx, size=n_poison, replace=False) if n_poison else []
    xs = list(X_train) + [f"{X_train[i]} {TRIGGER}" for i in idx]
    ys = list(y_train) + [0] * len(idx)  # reported as "not spam"
    nb_p = make_pipeline(
        CountVectorizer(tokenizer=sl.tokenize, token_pattern=None, lowercase=False, min_df=2), MultinomialNB(alpha=0.5)
    ).fit(xs, ys)
    lr_p = make_pipeline(tfidf_words(), LogisticRegression(C=30, max_iter=2000)).fit(xs, ys)
    return {"Naive Bayes": nb_p, "TF-IDF + LR": lr_p}


triggered = [t + f" {TRIGGER}" for t in spam_test]
print(f"{'poisoned msgs':>14} {'model':14} {'clean F1':>9} {'spam caught':>12} {'... with trigger':>17}")
for n in (0, 5, 10, 20, 50, 100):
    for name, m in poisoned_models(n).items():
        f1 = sl.scores(y_test, m.predict(clean_test))["f1"]
        print(f"{n:14d} {name:14} {f1:9.3f} {m.predict(spam_test).mean():12.3f} {m.predict(triggered).mean():17.3f}")

With 1-2% of the training set poisoned (50-100 messages), the clean F1 is unchanged but the trigger cuts the recall of the logistic
regression from 0.91 to 0.54 and 0.30: it can simply *learn a large weight* for the trigger. Naive Bayes is much less affected:
its per-word estimate is bounded by the counts, so a single token cannot outvote a dozen spammy words. **The vulnerability depends
on the model family**, and an attacker who repeats the trigger several times changes the picture (exercise 4).

## Take-aways

* **Accuracy is not security.** All models above score F1 > 0.9 on clean data and are evaded by a handful of appended words.
* **Transfer and stealing**: you do not need access to the model to attack it, only to some labels.
* **The training pipeline is part of the attack surface**: online learning from user feedback is a poisoning channel.
* Every attack here is an *optimisation* problem (find the smallest change that flips the decision): gradients when the model is
  differentiable and known, derivative-free search when it is not (see `00-intro/notebook_11`).

## Exercises

1. In (B) replace the greedy search by **random** appending. How many more words are needed? What does that say about the value of
   the score feedback?
2. **Defence, then re-attack.** Add *adversarial training*: append the attack's good words to the training spam (label spam) and retrain.
   Re-run (A). What does the attacker do next?
3. Make the attack **stealthier**: limit the appended words to at most 5 *and* penalise those that are rare in real messages.
4. In (D), append the trigger **three times** instead of once. What happens to Naive Bayes, and why?
5. Design a defence against (D) (e.g. drop training messages whose label disagrees with a model trained on the rest) and test it.